# Running the Eval

With a dataset in hand, we build the **core evaluation pipeline** — three small functions that take each test case, merge it into the prompt, send it to Claude, and (eventually) grade the result.

| Function | Responsibility |
|----------|----------------|
| `run_prompt(test_case)` | Merge the test case into the prompt template, call Claude, return the output |
| `run_test_case(test_case)` | Call `run_prompt`, then grade the output (grading is a **hardcoded 10** for now) |
| `run_eval(dataset)` | Loop over the whole dataset, collect all results |

The prompt under test is deliberately bare — no formatting instructions — so Claude returns **verbose** output. That's the baseline weakness we'll fix by iterating in later lessons.

> Runs on **Haiku 4.5** (same model the course uses for the pipeline). No prefilling here, so no flagship 400 — but we keep the `get_text()` helper and the explicit dataset path for consistency.

## Setup — client, model, helpers

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

import json
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5-20251001"


def get_text(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""


def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return get_text(message)


print(f"Ready. Model: {model}")

Ready. Model: claude-haiku-4-5-20251001


## `run_prompt` — merge a test case into the prompt and call Claude

The prompt is intentionally minimal (no formatting guidance), so expect verbose answers.

In [2]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

## `run_test_case` — run one case, then grade it

The grading is a placeholder `score = 10` for now. Replacing this with real grading logic is the whole point of the next two lessons (`Model based grading`, `Code based grading`).

In [3]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
    }

## `run_eval` — process the whole dataset

In [4]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

## Run it

Load the dataset we generated in the last lesson and run the pipeline. Expect ~30s even on Haiku — one API call per test case, sequentially. (Speeding this up comes later.)

In [5]:
DATASET_PATH = "01-building-with-the-claude-api/02-prompt-evaluation/dataset.json"

with open(DATASET_PATH, "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

## Examine the results

Each result is `{output, test_case, score}`. Note how verbose the `output` is — no formatting instructions yet, so Claude wraps answers in explanation. That's the exact weakness we'll grade against and then fix.

In [6]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Region Extraction Function\n\nHere's a comprehensive solution with multiple approaches:\n\n```python\nimport re\nfrom typing import Optional\n\ndef extract_region_from_s3_url(url: str) -> Optional[str]:\n    \"\"\"\n    Extract AWS region from an S3 bucket URL.\n    \n    Handles multiple S3 URL formats:\n    - s3.us-west-2.amazonaws.com\n    - s3-us-west-2.amazonaws.com\n    - bucket-name.s3.us-west-2.amazonaws.com\n    - bucket-name.s3-us-west-2.amazonaws.com\n    - s3.amazonaws.com (returns None for default region)\n    \n    Args:\n        url: S3 bucket URL string\n        \n    Returns:\n        Region code (e.g., 'us-west-2') or None if not found\n    \"\"\"\n    # Pattern 1: s3.REGION.amazonaws.com or s3-REGION.amazonaws.com\n    pattern1 = r's3[.-]([a-z0-9-]+)\\.amazonaws\\.com'\n    \n    match = re.search(pattern1, url)\n    if match:\n        region = match.group(1)\n        # Filter out bucket names that might match the pattern\n        if reg

## What we've got

That's the skeleton of essentially *every* eval pipeline: dataset → run through Claude → collect structured results. The one missing piece is the **grader** — the hardcoded `10` needs to become a real measurement. That's next: **model-based grading** (ask Claude to score) and **code-based grading** (deterministic checks).